# Machine Learning with Python - Code Examples

This notebook contains practical code examples for key machine learning concepts and workflows.

## Setup and Imports

First, let's import the libraries we'll need throughout this notebook.

In [ ]:
# Data manipulation libraries
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Machine learning libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix

# Set the style for matplotlib
plt.style.use('seaborn-whitegrid')
sns.set_palette('viridis')

## Data Loading and Exploration

Let's start with loading a sample dataset for car price prediction.

In [ ]:
# Sample code for loading car price dataset
# df = pd.read_csv('car_data.csv')

# For this example, we'll create a small synthetic dataset
np.random.seed(42)
n_samples = 200

# Create features
years = np.random.randint(2000, 2022, n_samples)
kilometers = np.random.randint(10000, 200000, n_samples)
engine_capacity = np.random.choice([1200, 1500, 1800, 2000, 2500, 3000], n_samples)
fuel_type = np.random.choice(['Petrol', 'Diesel', 'CNG', 'LPG'], n_samples, p=[0.5, 0.3, 0.1, 0.1])
transmission = np.random.choice(['Manual', 'Automatic'], n_samples)

# Create target variable (price) with some realistic relationship to features
base_price = 5000
year_effect = (years - 2000) * 100
km_effect = -0.02 * kilometers
engine_effect = engine_capacity * 0.5
fuel_effect = np.where(fuel_type == 'Diesel', 1000, 
                      np.where(fuel_type == 'Petrol', 500, 
                              np.where(fuel_type == 'CNG', 300, 0)))
transmission_effect = np.where(transmission == 'Automatic', 800, 0)
noise = np.random.normal(0, 1000, n_samples)

prices = base_price + year_effect + km_effect + engine_effect + fuel_effect + transmission_effect + noise
prices = np.maximum(prices, 1000)  # Ensure no negative prices

# Create dataframe
df = pd.DataFrame({
    'Year': years,
    'Kilometers_Driven': kilometers,
    'Engine_Capacity': engine_capacity,
    'Fuel_Type': fuel_type,
    'Transmission': transmission,
    'Price': prices
})

# Display the first few rows
df.head()

### Data Exploration

Let's explore our dataset to understand its characteristics.

In [ ]:
# Basic information about the dataset
print("Dataset shape:", df.shape)
print("\nDataset info:")
df.info()

# Summary statistics
print("\nSummary statistics:")
df.describe().round(2)

In [ ]:
# Visualize the distribution of the target variable (Price)
plt.figure(figsize=(10, 5))
sns.histplot(df['Price'], kde=True)
plt.title('Distribution of Car Prices')
plt.xlabel('Price')
plt.show()

# Visualize the relationship between Year and Price
plt.figure(figsize=(10, 5))
sns.scatterplot(x='Year', y='Price', data=df)
plt.title('Car Price vs. Year')
plt.show()

# Boxplot of Price by Fuel Type
plt.figure(figsize=(12, 6))
sns.boxplot(x='Fuel_Type', y='Price', data=df)
plt.title('Car Price by Fuel Type')
plt.show()

# Correlation matrix
numeric_cols = ['Year', 'Kilometers_Driven', 'Engine_Capacity', 'Price']
plt.figure(figsize=(10, 8))
correlation = df[numeric_cols].corr().round(2)
sns.heatmap(correlation, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()

## Data Preprocessing

Let's preprocess the data for machine learning.

In [ ]:
# Split features and target
X = df.drop('Price', axis=1)
y = df['Price']

# Identify categorical and numerical columns
categorical_cols = ['Fuel_Type', 'Transmission']
numerical_cols = ['Year', 'Kilometers_Driven', 'Engine_Capacity']

# Create preprocessing pipelines
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## Regression Models

Let's implement and evaluate different regression models for car price prediction.

### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

# Create a pipeline with preprocessing and model
linear_regression = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# Train the model
linear_regression.fit(X_train, y_train)

# Make predictions
y_pred_lr = linear_regression.predict(X_test)

# Evaluate the model
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Linear Regression Results:")
print(f"Mean Squared Error: {mse_lr:.2f}")
print(f"Root Mean Squared Error: {rmse_lr:.2f}")
print(f"R² Score: {r2_lr:.4f}")

### Regularized Regression: Ridge, Lasso, and ElasticNet

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# Ridge Regression
ridge = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0))
])

ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
r2_ridge = r2_score(y_test, y_pred_ridge)

# Lasso Regression
lasso = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Lasso(alpha=1.0))
])

lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
r2_lasso = r2_score(y_test, y_pred_lasso)

# ElasticNet Regression
elastic_net = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', ElasticNet(alpha=1.0, l1_ratio=0.5))
])

elastic_net.fit(X_train, y_train)
y_pred_elastic = elastic_net.predict(X_test)
r2_elastic = r2_score(y_test, y_pred_elastic)

# Compare the models
print("R² Scores for different regression models:")
print(f"Linear Regression: {r2_lr:.4f}")
print(f"Ridge Regression: {r2_ridge:.4f}")
print(f"Lasso Regression: {r2_lasso:.4f}")
print(f"ElasticNet Regression: {r2_elastic:.4f}")

### Random Forest Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Random Forest Regression
rf_regressor = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

rf_regressor.fit(X_train, y_train)
y_pred_rf = rf_regressor.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest Regression R² Score: {r2_rf:.4f}")

# Visualize predictions vs actual values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Random Forest: Predicted vs Actual Prices')
plt.show()